# Excel retrieval, step by step

Goal: find the rows that answer a question.
Each level adds one idea. Run top to bottom.
Last level saves embeddings for the agent's `semantic_row_search` tool.

## Setup

In [ ]:
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # find the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

from backend.agents.embeddings import Embedder
from backend.storage.s3 import configured_store

store = configured_store()  # local library, or S3 when DOCUMENTS_STORAGE=s3
embedder = Embedder()       # Titan Text Embeddings v2
_cache = {}

def embed_all(texts):
    """One vector per text. Same text is embedded once."""
    new = [t for t in dict.fromkeys(texts) if t not in _cache]
    with ThreadPoolExecutor(max_workers=8) as pool:
        for text, vector in zip(new, pool.map(embedder.embed, new)):
            _cache[text] = vector
    return np.array([_cache[t] for t in texts])

def search(chunks, vectors, query, k=5):
    """Top k chunks by similarity. Vectors are unit length, so dot product = cosine."""
    scores = vectors @ embed_all([query])[0]
    best = np.argsort(scores)[::-1][:k]
    return [(float(scores[i]), chunks[i]) for i in best]

def show(results, chars=300):
    for score, chunk in results:
        print(f"--- {score:.3f}\n{chunk[:chars]}\n")

FILENAME = "Location of 386 Pokémon in Gen 3.xlsx"  # must already be uploaded
SHEET = "Gen 3 Location Sheet"
QUERY = "Where can I find Ralts?"

from backend.storage.tables import load_table

doc = store.load(FILENAME)
sheets = doc.table_metadata["sheets"]
data_sheets = [name for name, meta in sheets.items() if meta["sheet_type"] == "data"]
print("Data sheets:", data_sheets)

def rows_of(sheet):
    """Preprocessed rows: header found, blank rows removed."""
    return load_table(doc.table_folder, sheets[sheet])

## Level 1: one row = one chunk, values only

Simplest. Each row becomes a comma-separated line.
Weakness: no column names. The model sees `280,..,Ralts,...` with no labels.

In [ ]:
table = rows_of(SHEET)
chunks = [", ".join(str(v) for v in row.values() if v not in (None, "")) for row in table.rows]
vectors = embed_all(chunks)
print(len(chunks), "chunks")
show(search(chunks, vectors, QUERY))

## Level 2: add column names

Each value gets its label: `NAME: Ralts`.
Labels give meaning. "Ruby" now means a game column.

In [ ]:
def labeled(row):
    return "\n".join(f"{col}: {val}" for col, val in row.items() if val not in (None, ""))

chunks = [labeled(row) for row in table.rows]
vectors = embed_all(chunks)
show(search(chunks, vectors, QUERY))

## Level 3: add sheet and row number

Each chunk says where it came from.
Needed to cite the source and check the answer.

In [ ]:
source_rows = sheets[SHEET]["source_rows"]  # Excel row numbers, after preprocessing

def located(sheet, row, number):
    return f"Sheet: {sheet} (row {number})\n{labeled(row)}"

chunks = [located(SHEET, row, source_rows[i]) for i, row in zip(table.index, table.rows)]
vectors = embed_all(chunks)
show(search(chunks, vectors, QUERY))

## Level 4: all data sheets

Answers can live on any tab.
Chunk every data sheet, search them together.

In [ ]:
chunks = []
for sheet in data_sheets:
    t = rows_of(sheet)
    numbers = sheets[sheet]["source_rows"]
    chunks += [located(sheet, row, numbers[i]) for i, row in zip(t.index, t.rows)]
vectors = embed_all(chunks)
print(len(chunks), "chunks across", len(data_sheets), "sheets")
show(search(chunks, vectors, QUERY))
level4_chunks, level4_vectors = chunks, vectors  # reused by a later level

## Level 5: row groups

Several rows per chunk, header once (CSV).
Fewer chunks, fewer embedding calls.
Trade-off: one chunk mixes topics, so matches get fuzzier.

In [ ]:
ROWS_PER_CHUNK = 10

import csv, io

def grouped(sheet):
    t = rows_of(sheet)
    numbers = sheets[sheet]["source_rows"]
    for start in range(0, len(t), ROWS_PER_CHUNK):
        part = t.rows[start:start + ROWS_PER_CHUNK]
        first, last = numbers[t.index[start]], numbers[t.index[start + len(part) - 1]]
        out = io.StringIO()
        writer = csv.writer(out, lineterminator="\n")
        writer.writerow(t.columns)
        writer.writerows([row.get(c, "") for c in t.columns] for row in part)
        yield f"Sheet: {sheet} (rows {first}-{last})\n{out.getvalue().strip()}"

chunks = [chunk for sheet in data_sheets for chunk in grouped(sheet)]
vectors = embed_all(chunks)
print(len(chunks), "chunks")
show(search(chunks, vectors, QUERY))

## Level 6: drop weak matches

Search always returns k results, even bad ones.
A minimum score drops the weak ones.
Pick it by looking at scores for good and bad queries.
Uses the level 4 chunks.

In [ ]:
MIN_SCORE = 0.35

def search_filtered(chunks, vectors, query, k=5, min_score=MIN_SCORE):
    return [(s, c) for s, c in search(chunks, vectors, query, k) if s >= min_score]

for q in [QUERY, "What is the capital of France?"]:
    print("Query:", q)
    show(search_filtered(level4_chunks, level4_vectors, q), chars=120)

## Level 7: save for the agent

Same idea as levels 3-5, using the backend code the agent uses.
Saves `row_embeddings.json` in this file's `preprocessed/` folder.
After this, the agent offers `semantic_row_search` for this spreadsheet.
Re-uploading the file deletes it. Run again after.

In [ ]:
CHUNKING = "row"  # "row" (levels 3-4) or "row_group" (level 5)

from backend.agents.embeddings import embed_rows
from backend.retrieval.semantic import row_search_documents

row_embeddings = embed_rows(doc, embedder, chunking=CHUNKING, rows_per_chunk=ROWS_PER_CHUNK)
print({k: v for k, v in row_embeddings["meta"].items() if k != "failed_chunks"})
store.save_row_embeddings(doc.storage_ref["manifest_key"], row_embeddings)

saved = store.load(FILENAME)  # read back what the agent will read
for m in row_search_documents({saved.id: saved}, QUERY, embedder, limit=5)["matches"]:
    print(f"--- {m['similarity']:.3f}  {m['sheet']} rows {m['source_rows']}\n{m['text'][:300]}\n")